In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_30_try_2.pickle

In [ ]:
%%RecordEvent
%%time
### cell 31 ###
def combine_ml_experience_data(question_of_interest, x_axis_title, include_2017=False):
    # original 2018 column name
    orig_col_2018 = 'For how many years have you used machine learning methods (at work or in school)?'
    # per-year category mappings
    category_mappings = {
        '2018': {
            '< 1 year': 'Under 1 year',
            '10-15 years': '10-20 years',
            '20+ years': '10-20 years',
            'I have never studied machine learning but plan to learn in the future': 'I do not use machine learning methods',
            'I have never studied machine learning and I do not plan to': 'I do not use machine learning methods',
        },
        '2019': {
            '< 1 years': 'Under 1 year',
            '10-15 years': '10-20 years',
            '20+ years': '20 or more years',
        }
    }
    # map of year → DataFrame
    year_dfs = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10,
    }
    if include_2017:
        year_dfs['2017'] = responses_df_2017

    dfs = []
    for year, df in year_dfs.items():
        df_year = df
        # rename 2018 column to the standard question name
        if year == '2018':
            df_year = df_year.rename(columns={orig_col_2018: question_of_interest})
        # apply any category-level normalization
        mapping = category_mappings.get(year)
        if mapping:
            df_year = df_year.copy()
            df_year[question_of_interest] = df_year[question_of_interest].replace(mapping)
        # compute percentage distribution
        vc = df_year[question_of_interest].value_counts(dropna=False)
        pct = (vc / vc.sum() * 100).round(1).reset_index()
        pct.columns = [x_axis_title, 'percentage']
        pct['year'] = year
        dfs.append(pct)

    combined = pd.concat(dfs, ignore_index=True)
    # ensure column order remains ['percentage', 'year', x_axis_title]
    return combined[['percentage', 'year', x_axis_title]]

# parameters
question_of_interest_cell43 = 'For how many years have you used machine learning methods?'
# combine and sort
ml_exp_df_combined = combine_ml_experience_data(
    question_of_interest_cell43,
    title_for_x_axis_cell43
).sort_values(['year', 'percentage']).reset_index(drop=True)
ml_exp_df_combined.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_31_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_31_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[31], f)


In [ ]:
opt_output = Out.get(4)